In [96]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [97]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rahuljangir78/esg-and-financial-metrics-of-manufacturing-firms")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/rahuljangir78/esg-and-financial-metrics-of-manufacturing-firms


In [98]:
df = pd.read_csv("/kaggle/input/datasets/rahuljangir78/esg-and-financial-metrics-of-manufacturing-firms/Manufacturing_ESG_Financial_Data.csv")

In [99]:
df.columns

Index(['Firm_ID', 'Year', 'Industry_Type', 'ESG_Score', 'E_Score', 'S_Score',
       'G_Score', 'ROA', 'ROE', 'Net_Profit_Margin', 'Revenue',
       'Operating_Cost', 'Firm_Size', 'Board_Independence',
       'Innovation_Spending'],
      dtype='object')

In [108]:
df["E_Compliance"] = df["E_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")
df["S_Compliance"] = df["S_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")
df["G_Compliance"] = df["G_Score"].apply(lambda x: "Violation" if x < 60 else "Compliant")


In [109]:
df["Overall_Compliance"] = df.apply(
    lambda row: "Non-Compliant"
    if "Violation" in [row["E_Compliance"], row["S_Compliance"], row["G_Compliance"]]
    else "Compliant",
    axis=1
)


In [111]:
df["compliance_risk_score"] = (
    (df["E_Compliance"] == "Violation").astype(int) +
    (df["S_Compliance"] == "Violation").astype(int) +
    (df["G_Compliance"] == "Violation").astype(int)
) * 33


In [112]:
df["performance_risk"] = (100 - df["ESG_Score"])


In [114]:
df["final_esg_risk_score"] = (
    df["performance_risk"] * 0.6 +
    df["compliance_risk_score"] * 0.4
)


In [118]:
def classify_alert(score):
    if score >= 70:
        return "Critical"
    elif score >= 40:
        return "Warning"
    else:
        return "Low Risk"

df["alert_level"] = df["final_esg_risk_score"].apply(classify_alert)


In [119]:
agent4_output = df[[
    "Firm_ID",
    "Year",
    "ESG_Score",
    "Overall_Compliance",
    "final_esg_risk_score",
    "alert_level"
]]


In [120]:
agent4_output.head()

,Firm_ID,Year,ESG_Score,Overall_Compliance,final_esg_risk_score,alert_level
0,MFG0103,2023,64.746667,Compliant,21.152,Low Risk
1,MFG0436,2021,78.060000,Compliant,13.164,Low Risk
2,MFG0349,2023,66.936667,Non-Compliant,33.038,Low Risk
3,MFG0271,2021,74.530000,Non-Compliant,28.482,Low Risk
4,MFG0107,2019,68.150000,Non-Compliant,32.310,Low Risk


In [121]:
AGENT4_output.to_csv("agent4_output.csv", index=False)

In [125]:
AGENT4_json = AGENT4_output.to_json(orient="records")

In [126]:
AGENT4_json


'[{"E_Compliance":"Compliant","S_Compliance":"Compliant","G_Compliance":"Compliant","Overall_Compliance":"Compliant","compliance_risk_score":0,"alert_level":"Low Risk"},{"E_Compliance":"Compliant","S_Compliance":"Compliant","G_Compliance":"Compliant","Overall_Compliance":"Compliant","compliance_risk_score":0,"alert_level":"Low Risk"},{"E_Compliance":"Compliant","S_Compliance":"Violation","G_Compliance":"Compliant","Overall_Compliance":"Non-Compliant","compliance_risk_score":33,"alert_level":"Warning"},{"E_Compliance":"Compliant","S_Compliance":"Compliant","G_Compliance":"Violation","Overall_Compliance":"Non-Compliant","compliance_risk_score":33,"alert_level":"Warning"},{"E_Compliance":"Violation","S_Compliance":"Compliant","G_Compliance":"Compliant","Overall_Compliance":"Non-Compliant","compliance_risk_score":33,"alert_level":"Warning"},{"E_Compliance":"Compliant","S_Compliance":"Compliant","G_Compliance":"Compliant","Overall_Compliance":"Compliant","compliance_risk_score":0,"alert_lev